# Lecture 08, Notebook 10: Benchmark OLG DEQN, Persistent Simulation in Lux

This variant rolls a state cloud through the benchmark OLG policy transform,
preserving the borrowing and collateral constraints in the shared helper.

## Lecture 08, Notebook 10: 56-Agent OLG Benchmark — DEQN (Persistent Simulation)

**Course:** Deep Learning for Solving and Estimating Dynamic Models in Economics and Finance
**Script reference:** §5.5 (the 56-agent Azinovic–Gaegauf–Scheidegger 2022 benchmark; two assets, borrowing/collateral constraints, persistent shocks); §5.3–§5.4 (DEQN mapping, product-form KKT)
**Notebook role:** core
**Author:** Simon Scheidegger

*Julia/Lux preview of* `lectures/lecture_08_olg_models_deqns/code/lecture_08_10_OLG_Benchmark_DEQN_persistent.ipynb`.

> **Run mode.** The checked-in run uses `RUN_MODE = "smoke"` with fixed `SEED = 0`. In smoke mode the 56-cohort problem is **reduced to `n_ages = 8`** for a fast structural check — this is not convergence or production validation. `"teaching"` and `"production"` restore the full $A = 56$ benchmark and raise the segment, batch, and evaluation counts, but this Julia preview keeps the same compact `(32, 32)` `tanh` network and fixed `Adam(0.001)` optimizer in every mode; unlike Python, it does not switch production to a ~1000-unit network or a two-stage learning-rate schedule.

> **Sampling mode.** This is the primary classroom variant: the training cloud is **generated by simulating** the economy under the current policy, each segment continuing from the previous one — the production setting in AGS (2022). The feedback-free ablation is `lecture_08_09_OLG_Benchmark_DEQN_exogenous_Lux.ipynb`.

This notebook solves the 56-period overlapping-generations benchmark of Azinovic, Gaegauf and Scheidegger (2022): risky illiquid capital, a risk-free one-period bond, cohort-specific borrowing and collateral constraints, lifecycle labor income, and aggregate TFP / depreciation shocks. The equilibrium conditions are the cohort-stacked capital and bond Euler equations, product-form KKT complementarity residuals for the borrowing and collateral inequalities ($\lambda_b^h\,k'^h$ and $\mu^h\,q^h$, both squared in the loss), and an explicit bond-market clearing residual. This is the primary classroom notebook for the 56-agent benchmark.

The notebook mirrors the structure of the IRBC and analytic-OLG notebooks: parameters, augmented state, neural-network policy transform, residual / loss construction, training-data generation, compact full-batch segment training, out-of-sample residual tables, and policy-stability checks. This is a teaching preview, not a full replacement for the Python benchmark artifact.

In [ ]:
import Pkg
Pkg.activate(joinpath(@__DIR__, "..", "..", "..", "julia"))

using DLEFJulia
using Lux
using NNlib
using Optimisers
using Statistics

In [ ]:
RUN_MODE = "smoke"
SEED = 0
budgets = (
    smoke = (segments = 3, batch_size = 8, segment_length = 4, anchor_size = 16, eval_burn_in = 4, eval_length = 4, n_ages = 8),
    teaching = (segments = 60, batch_size = 64, segment_length = 8, anchor_size = 256, eval_burn_in = 32, eval_length = 64, n_ages = 56),
    production = (segments = 2_000, batch_size = 256, segment_length = 16, anchor_size = 2_048, eval_burn_in = 128, eval_length = 256, n_ages = 56),
)
hp = run_mode_budget(RUN_MODE; budgets)
rng = rng_from_seed(SEED)

## 1. Economic parameters

The Azinovic–Gaegauf–Scheidegger (2022) benchmark calibration: $A = 56$ overlapping agents (ages 25–80), Cobb–Douglas production, a lifecycle labor profile, cohort-specific borrowing and collateral constraints, convex capital adjustment costs, and four persistent (Markov) aggregate-shock states combining TFP $\eta$ and depreciation $\delta$. Here `BenchmarkOLGParams(n_ages = hp.n_ages)` carries this calibration; in smoke mode `hp.n_ages = 8`. `cloud = sample_benchmark_olg_states(rng, params, hp.batch_size)` draws the **initial random feasible trajectory heads** that the simulation will roll forward.

### 2. State representation and augmented network inputs

The minimal state is $(z_t, \mathbf{k}_t, \mathbf{b}_t)$ — the aggregate shock index plus the cohort capital and bond holdings — of dimension $1 + 2A = 113$ at $A = 56$. For learning, the network is fed an engineered state of dimension $240$,

$$
  \mathbf{s}_t = \bigl(z_t,\ \mathrm{onehot}(z_t),\ \eta_t,\ \delta_t,\ K_t,\ L_t,\ r_t,\ w_t,\ Y_t,\ \mathbf{k}_t,\ \mathrm{fininc}_t,\ \mathrm{labinc}_t,\ \mathrm{cash}_t,\ \boldsymbol{\pi}(z_t,\cdot)\bigr) \in \mathbb{R}^{240},
$$

where $\mathrm{fininc}_t^h = r_t k_t^h + b_t^h$, $\mathrm{labinc}_t^h = w_t e^h$, $\mathrm{cash}_t^h = r_t k_t^h + b_t^h + w_t e^h$, and $Y_t = \eta_t K_t^\alpha L_t^{1-\alpha} + (1-\delta_t)K_t$. The bond vector is recoverable from financial income and is not passed separately; the bond price is a network **output**. In Julia the feature width is `benchmark_olg_feature_dim(params)` (smaller in smoke mode); the augmentation is feature engineering only.

### 3. Neural network and policy transformation

The policy network outputs four cohort-specific blocks plus the bond price:

$$
  \mathcal{N}_\theta(\mathbf{s}_t) \;\longrightarrow\; \bigl(\hat{k}_{t+1}^h,\, \hat{\lambda}_b^{h},\, \hat{q}^h,\, \hat{\mu}^h\bigr)_{h=1}^{A-1} \;\cup\; \{\hat{q}_t\},
$$

for a total of $4(A-1) + 1$ outputs (`raw_dim = 4 * (params.n_ages - 1) + 1`). Softplus heads enforce $\hat{k}_{t+1}^h, \hat{\lambda}_b^h, \hat{\mu}^h \ge 0$ by construction (the bond holding is recovered as $\hat{b}_{t+1}^h = (\hat{q}^h - \hat{k}_{t+1}^h)/\kappa$); the inequalities are then enforced softly via the product-form KKT residuals of §4. In Lux the network is `make_mlp(benchmark_olg_feature_dim(params), (32, 32), raw_dim; activation = NNlib.tanh)`, with the policy transform in `benchmark_olg_policy_from_raw` / `benchmark_olg_residual` and `Optimisers.Adam` on Float64 parameters.

In [ ]:
params = BenchmarkOLGParams(n_ages = hp.n_ages)
raw_dim = 4 * (params.n_ages - 1) + 1
model = make_mlp(benchmark_olg_feature_dim(params), (32, 32), raw_dim; activation = NNlib.tanh)
state = setup_training(rng_from_seed(SEED; offset = 1), model, Optimisers.Adam(0.001); parameter_type = Float64)
cloud = sample_benchmark_olg_states(rng, params, hp.batch_size)

## 4. Residuals and loss

For each cohort $h = 1, \ldots, A-1$ there is a capital Euler, a bond Euler, and two complementarity residuals. The capital Euler in relative form is

$$
  \mathrm{EulerErr}_h^{(k)}(\mathbf{s}_t) \;=\;
  \frac{\beta\, \mathbb{E}_t\!\left[\, u'(c_{t+1}^{h+1})\, \bigl(R_{t+1} + 1 - \delta_{t+1}\bigr)\,\right]}
       {u'(c_t^h)\,\bigl(1 + \zeta(k_{t+1}^h - r\,k_t^h)\bigr)} \;-\; 1,
$$

with the analogous residual for the bond. Borrowing- and collateral-constraint complementarity is enforced softly by adding *squared product residuals*:

$$
  \mathrm{KKT}_{\mathrm{borrow}}^h \;=\; \lambda_b^h \cdot k'^h, \qquad
  \mathrm{KKT}_{\mathrm{coll}}^h \;=\; \mu^h \cdot q^h.
$$

Both vanish exactly at any KKT point and have a differentiable squared form. (The Fischer–Burmeister alternative $\Phi(a,b) = a + b - \sqrt{a^2 + b^2}$ is used in the IRBC notebook of Chapter 3, where the irreversibility constraint binds more often.) Bond-market clearing is enforced as a residual, $\sum_{h} b_{t+1}^h = 0$. The total loss is the mean-squared sum of all $4(A-1) + 1$ residuals, with `KKT_WEIGHT` and `MARKET_WEIGHT` balancing the scales. In Julia this is `benchmark_olg_residual`, wrapped by `benchmark_loss`.

> The full Python notebook also adds two feasibility penalties to the loss — one for negative consumption, one for bad capital-adjustment factors — scaled by `PENALTY_WEIGHT = 10.0`. The shared Julia helper keeps the same two feasibility penalties but at unit weight, so short smoke trials here tolerate infeasible ($c \le 0$, negative adjustment-factor) draws somewhat more before penalizing them.

### Training-data construction, training loop, and policy-stability

In this persistent version the training states are **generated by simulating** the economy under the current network policy: initial feasible heads, then each segment continues from the terminal states of the previous one.

The next cell defines the machinery and runs the segment loop:

- `draw_benchmark_next_shocks` samples the next aggregate shock from the Markov transition row, and `simulate_benchmark_segment` rolls the economy forward under the current policy via `benchmark_olg_next_states`, applying the borrowing/collateral transforms and returning the segment path and terminal states.
- The loop feeds each segment once to `train_step!` as a full-batch Adam step (`max_grad_norm = 10`), then sets `cloud_head = cloud_end` — the explicit realization of the Python `X_segment, X_end = get_training_segment(...); X_start = X_end` continuation.
- **Policy-stability (§7).** `benchmark_policy_fingerprint` evaluates the policy on a fixed holdout `anchor_states` and `relative_policy_drift_stats` reports `policy_drift_rms` / `policy_drift_max` after each segment; `benchmark_state_collateral_slack` tracks how close the simulated states sit to the collateral constraint.

> The full Python notebook runs `PASSES_PER_SEGMENT` passes over shuffled mini-batches from each segment. This preview takes one full-batch Adam step per segment, so `hp.batch_size` is the number of simulated tracks rather than an SGD mini-batch size.

> Python also reports a same-state repeatability check and PASS/WARNING verdict against `TIME_INVARIANCE_TOL_RMS` / `TIME_INVARIANCE_TOL_MAX`. This preview returns raw `policy_drift_rms` / `policy_drift_max` statistics only.

> The full Python notebook also runs a bad-state repair inside its `simulate_path` — flagging numerically invalid states (non-finite entries, aggregate capital outside its bounds, or collateral violations) and replacing them with fresh feasible-box draws. This preview omits the repair: the borrowing/collateral transforms already guarantee $k > 0$ and non-negative collateral slack, so short smoke segments stay bounded without it.

In [ ]:
function draw_benchmark_next_shocks(rng, states; params)
    z_current = clamp.(round.(Int, vec(states[1, :])), 1, length(params.tfp))
    z_next = similar(z_current)
    for (i, z) in pairs(z_current)
        u = rand(rng)
        cumulative = zero(eltype(params.transition))
        z_next[i] = length(params.tfp)
        for shock in eachindex(params.tfp)
            cumulative += params.transition[z, shock]
            if u <= cumulative
                z_next[i] = shock
                break
            end
        end
    end
    return z_next
end

function simulate_benchmark_segment(rng, train_state, start_states, steps; params)
    n_tracks = size(start_states, 2)
    path = similar(start_states, size(start_states, 1), n_tracks * steps)
    current = copy(start_states)
    for t in 1:steps
        cols = ((t - 1) * n_tracks + 1):(t * n_tracks)
        path[:, cols] .= current
        z_next = draw_benchmark_next_shocks(rng, current; params)
        current = benchmark_olg_next_states(
            train_state.model, train_state.ps, train_state.st, current, z_next; params
        )
    end
    return path, current
end

function benchmark_policy_fingerprint(train_state, states; params)
    raw, _ = train_state.model(benchmark_olg_features(states; params), train_state.ps, train_state.st)
    policy = benchmark_olg_policy_from_raw(raw, states; params)
    return vcat(
        log1p.(policy.capital),
        policy.bond ./ (1 .+ abs.(policy.bond)),
        log.(max.(policy.lambda, params.consumption_floor)),
        log.(max.(policy.mu, params.consumption_floor)),
        log.(max.(policy.price, params.consumption_floor) ./ params.beta),
    )
end

function relative_policy_drift_stats(previous, current)
    diff = current .- previous
    scale = 1 + sqrt(mean(abs2, previous))
    return (
        rms = sqrt(mean(abs2, diff)) / scale,
        max_abs = maximum(abs.(diff)) / scale,
    )
end

function benchmark_shock_counts(states; params)
    z = clamp.(round.(Int, vec(states[1, :])), 1, length(params.tfp))
    return [count(==(shock), z) for shock in eachindex(params.tfp)]
end

function benchmark_state_collateral_slack(states; params)
    k = @view states[2:(1 + params.n_ages), :]
    b = @view states[(2 + params.n_ages):(1 + 2 * params.n_ages), :]
    return minimum(b .+ k ./ params.kappa)
end

benchmark_loss(model, ps, st, batch) = begin
    pieces, st_new = benchmark_olg_residual(model, ps, st, batch; params)
    return pieces.loss, st_new
end

train_result = let cloud_head = cloud,
        sim_rng = rng_from_seed(SEED; offset = 10_000),
        anchor_states = sample_benchmark_olg_states(rng_from_seed(SEED; offset = 999_001), params, hp.anchor_size)
    initial_loss_local = loss_value(state, benchmark_loss, cloud_head)
    history_local = NamedTuple[]
    anchor_previous = benchmark_policy_fingerprint(state, anchor_states; params)
    last_segment_local = cloud_head
    for segment in 1:hp.segments
        segment_states, cloud_end = simulate_benchmark_segment(sim_rng, state, cloud_head, hp.segment_length; params)
        metrics = train_step!(state, benchmark_loss, segment_states; max_grad_norm = 10.0)
        monitor, _ = benchmark_olg_residual(state.model, state.ps, state.st, segment_states; params)
        anchor_current = benchmark_policy_fingerprint(state, anchor_states; params)
        drift = relative_policy_drift_stats(anchor_previous, anchor_current)
        anchor_previous = anchor_current
        append_metric!(
            history_local;
            segment,
            loss = metrics.loss,
            monitor_loss = monitor.loss,
            policy_drift_rms = drift.rms,
            policy_drift_max = drift.max_abs,
            mean_abs_capital_euler = residual_summary(monitor.euler_capital).mean_abs,
            mean_abs_bond_euler = residual_summary(monitor.euler_bond).mean_abs,
            shock_counts = benchmark_shock_counts(segment_states; params),
        )
        cloud_head = cloud_end
        last_segment_local = segment_states
    end
    (initial_loss = initial_loss_local, history = history_local, cloud = cloud_head, last_segment = last_segment_local)
end
initial_loss = train_result.initial_loss
history = train_result.history
cloud_final = train_result.cloud
last_segment = train_result.last_segment

## 6. Final diagnostics

The diagnostics report the constraint- and equilibrium-condition residuals on two out-of-sample clouds — the **last training segment** and a **fresh simulated ergodic cloud** (`simulate_benchmark_segment` with `eval_burn_in` states dropped): max absolute capital and bond Euler residuals, minimum collateral slack, and mean absolute bond-market residual. This gives accuracy on the region induced by the learned policy. The simulated eval cloud uses `hp.batch_size` tracks; Python has a separate `EVAL_TRACKS` setting, but this preview ties eval-track count to the training-track budget.

> The full Python notebook also draws lifecycle plots (cohort capital/consumption by shock state and the bond-market residual along a long simulated trajectory), replaced here by the returned diagnostics. It further reports the same residuals on a third cloud — an **exogenous off-trajectory test cloud** (fresh `sample_benchmark_olg_states` draws) for robustness away from the ergodic set. That off-trajectory diagnostic is dropped here; the feedback-free companion notebook `lecture_08_09_OLG_Benchmark_DEQN_exogenous` trains and evaluates on exactly such an exogenous cloud.

In [ ]:
diagnostics, _ = benchmark_olg_residual(state.model, state.ps, state.st, last_segment; params)
eval_start = sample_benchmark_olg_states(rng_from_seed(SEED; offset = 777), params, hp.batch_size)
eval_path, _ = simulate_benchmark_segment(
    rng_from_seed(SEED; offset = 888), state, eval_start, hp.eval_burn_in + hp.eval_length; params
)
eval_sim = eval_path[:, (hp.eval_burn_in * hp.batch_size + 1):end]
eval_diagnostics, _ = benchmark_olg_residual(state.model, state.ps, state.st, eval_sim; params)

## Conclusion

This Lux preview trained the 56-cohort AGS (2022) benchmark DEQN on its **own simulated ergodic set**: two assets, softplus policy heads, product-form KKT residuals for the borrowing/collateral inequalities, bond-market clearing, segment-continuation simulation, and policy-drift monitoring.

In smoke mode this runs at `n_ages = 8` as a **structural check, not a convergence run**; set `RUN_MODE = "teaching"`/`"production"` to restore the full $A = 56$ benchmark. The cell below returns a machine-checkable summary (cohort count, initial/final loss, capital/bond Euler residuals on the last segment and the simulated eval cloud, collateral slack, bond-market residuals, and policy-drift statistics) for this run.

In [ ]:
(
    sampling = :persistent_simulation,
    transition = :markov_policy_simulation,
    n_ages = params.n_ages,
    states_per_segment = hp.batch_size * hp.segment_length,
    initial_loss = initial_loss,
    final_loss = history[end].monitor_loss,
    simulated_eval_loss = eval_diagnostics.loss,
    max_abs_capital_euler = residual_summary(diagnostics.euler_capital).max_abs,
    max_abs_bond_euler = residual_summary(diagnostics.euler_bond).max_abs,
    simulated_eval_max_abs_capital_euler = residual_summary(eval_diagnostics.euler_capital).max_abs,
    simulated_eval_max_abs_bond_euler = residual_summary(eval_diagnostics.euler_bond).max_abs,
    min_collateral_slack = minimum(diagnostics.collateral),
    min_state_collateral_slack = benchmark_state_collateral_slack(last_segment; params),
    simulated_eval_min_state_collateral_slack = benchmark_state_collateral_slack(eval_sim; params),
    mean_abs_bond_market = mean(abs, diagnostics.bond_market),
    simulated_eval_mean_abs_bond_market = mean(abs, eval_diagnostics.bond_market),
    last_policy_drift_rms = history[end].policy_drift_rms,
    last_policy_drift_max = history[end].policy_drift_max,
    last_segment_shock_counts = history[end].shock_counts,
    simulated_eval_shock_counts = benchmark_shock_counts(eval_sim; params),
)